In [1]:
%load_ext autoreload
%autoreload 2
import polars as pl
from pathlib import Path
from src.utils import load_jsonld, unique_taxonomy_schemes
from src.text_analyze import get_corpus_by_lang
import numpy as np

# Loading data

In [2]:
DATA = Path("./data")

In [3]:
tax_docs = load_jsonld(DATA / "taxonomy_labels.json")
wp_docs = load_jsonld(DATA / "working_papers_metadata.json")

# Analysis

Regarding the taxonomy dataset, I found that it contains different types of schemas. Containing metadata from authors, countries, departments, topics and more. 

In [4]:
unique_taxonomy_schemes(tax_docs)

{'http://thesaurus.iadb.org/idbthesauri/05209516-fd4c-445d-924d-b71fe09b21b5',
 'http://thesaurus.iadb.org/idbthesauri/59f0f940-43f5-4cad-a424-d677de762c8c',
 'http://thesaurus.iadb.org/idbthesauri/5dc9c443-6a9c-41f4-8550-47012e2ba3c9',
 'http://thesaurus.iadb.org/idbthesauri/6a09e747-27c8-4b31-b61b-a761e95f0db7',
 'http://thesaurus.iadb.org/idbthesauri/Discontinued-Departments',
 'http://thesaurus.iadb.org/idbthesauri/IdBAuthors',
 'http://thesaurus.iadb.org/idbthesauri/IdBCountries',
 'http://thesaurus.iadb.org/idbthesauri/IdBDepartments',
 'http://thesaurus.iadb.org/idbthesauri/IdBInstitutions',
 'http://thesaurus.iadb.org/idbthesauri/IdBRegions',
 'http://thesaurus.iadb.org/idbthesauri/IdBTopics',
 'http://thesaurus.iadb.org/idbthesauri/ab28388e-71fa-4117-a8b5-34e6b8eb3d00',
 'https://taxonomy.iadb.org/jelcodes/9a582715-b63b-4961-b0c7-8f413ce81b41'}

Some taxonomy refers to:
- Type of operation (credit, technical cooperation) of the document. For example, 'http://thesaurus.iadb.org/idbthesauri/5dc9c443-6a9c-41f4-8550-47012e2ba3c9' and  'http://thesaurus.iadb.org/idbthesauri/05209516-fd4c-445d-924d-b71fe09b21b5'.
- Departments (current and discountinued). For example, 'http://thesaurus.iadb.org/idbthesauri/Discontinued-Departments' and 'http://thesaurus.iadb.org/idbthesauri/IdBDepartments', 'http://thesaurus.iadb.org/idbthesauri/IdBInstitutions'
- Authors. 'http://thesaurus.iadb.org/idbthesauri/IdBAuthors'
- Countries or Regions. 'http://thesaurus.iadb.org/idbthesauri/IdBCountries', 'http://thesaurus.iadb.org/idbthesauri/IdBRegions'
- Topics. 'http://thesaurus.iadb.org/idbthesauri/IdBTopics', 'https://taxonomy.iadb.org/jelcodes/9a582715-b63b-4961-b0c7-8f413ce81b41', 'http://thesaurus.iadb.org/idbthesauri/ab28388e-71fa-4117-a8b5-34e6b8eb3d00'


In [5]:
taxonomy =  'http://thesaurus.iadb.org/idbthesauri/ab28388e-71fa-4117-a8b5-34e6b8eb3d00'
for tax in tax_docs:
    try:
        if tax["skos:inScheme"]["@id"] == taxonomy: 
            print(tax)
    except TypeError:
        for scheme in tax["skos:inScheme"]:
            if scheme["@id"] == taxonomy: 
                print(tax)
    except KeyError:
        continue

{'@id': 'http://thesaurus.iadb.org/idbthesauri/4e3f6d2e-23ab-47b6-b67c-f03941dd4fa1', 'skos:prefLabel': [{'@value': 'Mangrove Ecosystems', '@language': 'en'}, {'@value': 'Ecossistema de Mangue', '@language': 'pt'}, {'@value': 'Écosystème de Mangrove', '@language': 'fr'}, {'@value': 'Ecosistema de Mangle', '@language': 'es'}], 'skos:inScheme': [{'@id': 'http://thesaurus.iadb.org/idbthesauri/IdBTopics'}, {'@id': 'http://thesaurus.iadb.org/idbthesauri/ab28388e-71fa-4117-a8b5-34e6b8eb3d00'}]}
{'@id': 'http://thesaurus.iadb.org/idbthesauri/7ffef416-2263-48ec-ad2a-7cbc6745ce75', 'skos:prefLabel': [{'@value': 'Household Expenditure', '@language': 'en'}, {'@value': 'Gasto Familiar', '@language': 'es'}, {'@value': 'Dépense Ménagère', '@language': 'fr'}, {'@value': 'Despesa Familiar', '@language': 'pt'}], 'skos:inScheme': [{'@id': 'http://thesaurus.iadb.org/idbthesauri/IdBTopics'}, {'@id': 'http://thesaurus.iadb.org/idbthesauri/ab28388e-71fa-4117-a8b5-34e6b8eb3d00'}]}
{'@id': 'http://thesaurus.i

Schema

Analysis to check if all the docs have the same schema. 

- Not all the documents have the same number of attributes.
- There are three schema attributes that don't appear in all documents' metadata: keywords, jelCode and spatialCoverage

In [6]:
n_keys_dict = {}
for doc in wp_docs:
    len_keys = len(doc.keys())

    n_keys_dict[len_keys] = n_keys_dict.get(len_keys, 0) + 1

n_keys_dict

{25: 29, 26: 113, 24: 7, 23: 1}

In [7]:
keys_count = {}

for doc in wp_docs:
    for key in doc.keys():
        keys_count[key] = keys_count.get(key, 0) + 1

for key, val in keys_count.items():
    if val != 150:
        print(f"{key} only found in {val}/150")

schema:keywords only found in 130/150
idb:jelCode only found in 149/150
schema:spatialCoverage only found in 125/150


In [8]:
from pyld import jsonld
import pandas as pd
flattened_data = jsonld.flatten(wp_docs)

In [9]:
df = pd.json_normalize(flattened_data)

In [10]:
df.columns

Index(['@id', '@type', 'idb:jelCode', 'idb:knowledgeProductCategory',
       'idb:operation', 'idb:orgUnit', 'idb:peerReviewed', 'idb:sourceCatalog',
       'schema:about', 'schema:author', 'schema:dateCreated',
       'schema:dateModified', 'schema:datePublished', 'schema:description',
       'schema:encoding', 'schema:inLanguage', 'schema:keywords',
       'schema:license', 'schema:mainEntityOfPage', 'schema:name',
       'schema:numberOfPages', 'schema:publisher', 'schema:sameAs',
       'schema:spatialCoverage', 'schema:text', 'schema:url'],
      dtype='str')

In [11]:
# Checking if all ids starts with the IADB web domain
sum([id.startswith("https://publications.iadb.org/") for id in df['@id'].to_list()])

150

In [12]:
# Checking that all documents are WorkingPapers
{(item[0], item[1]) for item in df['@type'].to_list()}

{('idb:WorkingPaper', 'idb:MainPublication')}

In [13]:
# Checking that all documents with JEL Code refers to the JELCODES taxonomy
np.mean([code["@id"].startswith('https://taxonomy.iadb.org/jelcodes/') for doc in df['idb:jelCode'].to_list() if isinstance(doc, list) for code in doc ])

np.float64(1.0)

In [14]:
# Checking that all documents refers to the same knowledge product category
{item[0]["@id"] for item in df['idb:knowledgeProductCategory'].to_list()}

{'http://thesaurus.iadb.org/idbthesauri/cf4f3969-8d0f-404d-bd9b-c27b2c1c60d8'}

In [15]:
# Checking that all documents' operation refers to operation taxonomy
np.mean([code["@id"].startswith('https://convergence.iadb.org/operation/') for doc in df['idb:operation'].to_list() if isinstance(doc, list) for code in doc ])


np.float64(1.0)

In [16]:
# Checking that all orgUnit refers to the same taxonomy
np.mean([code["@id"].startswith('http://thesaurus.iadb.org/idbthesauri/') for doc in df['idb:orgUnit'].to_list() if isinstance(doc, list) for code in doc ])

np.float64(1.0)

In [17]:
# Checking that all working papers are peer reviewed
{doc[0]["@value"] for doc in df['idb:peerReviewed'].to_list()}

{True}

In [18]:
# Checking source
{doc[0]["@id"] for doc in df['idb:sourceCatalog'].to_list()}

{'https://publications.iadb.org'}

In [19]:
# Checking that all 'about' entries refers to the same taxonomy
np.mean([code["@id"].startswith('http://thesaurus.iadb.org/idbthesauri/') for doc in df['schema:about'].to_list() if isinstance(doc, list) for code in doc ])

np.float64(1.0)

In [20]:
# Checking that all 'authors' entries refers to the same taxonomy
np.mean([code["@id"].startswith('http://thesaurus.iadb.org/idbthesauri/') for doc in df['schema:author'].to_list() if isinstance(doc, list) for code in doc ])

np.float64(1.0)

In [21]:
# Checking that all 'dateCreated' entries are dates
np.mean([code["@type"] == 'http://www.w3.org/2001/XMLSchema#date' for doc in df['schema:dateCreated'].to_list() if isinstance(doc, list) for code in doc ])

np.float64(1.0)

In [22]:
# Checking that all 'dateCreated' entries are dateTime
np.mean([code["@type"] == 'http://www.w3.org/2001/XMLSchema#dateTime' for doc in df['schema:dateModified'].to_list() if isinstance(doc, list) for code in doc ])

np.float64(1.0)

In [23]:
# Checking that all 'datePublished' entries are dates
np.mean([code["@type"] == 'http://www.w3.org/2001/XMLSchema#date' for doc in df['schema:datePublished'].to_list() if isinstance(doc, list) for code in doc ])

np.float64(1.0)

Main problem in the WP dataset

In [24]:
docs_keywords = [doc[0]["@value"] for doc in df['schema:keywords'].to_list() if isinstance(doc, list) ]

In [25]:
unique_keywords = set()
count_keywords = 0
for doc in docs_keywords:
    kw = doc.split(";")
    unique_keywords.update(kw)
    count_keywords += len(kw)

print(f"There are {len(unique_keywords)} unique keywords in a total of 150 documents tagged with a total of {count_keywords}")

There are 501 unique keywords in a total of 150 documents tagged with a total of 617


## Accuracy analysis

In [26]:
# Extracting the ids, description, text and aboutness of the workiing papers dataset.
corpus = get_corpus_by_lang(wp_docs, tax_docs)

In [27]:
corpus_en = [(idx, content['desc'], content['text'], content['about'], content['keywords']) for idx, content in corpus.items() if content['lang'] == 'en']
corpus_es = [(idx, content['desc'], content['text'], content['about'], content['keywords']) for idx, content in corpus.items() if content['lang'] == 'es']

#### Analyis using ChromaDB
In this analysis, I'm creating a ChromaDB to create a semantic search engine to analyze if the 'about' tags that are used in the working papers are accurate. As I'm working with a semantic search, I will expect that the 'about' tags are descriptive enough to identify the working paper by querying the database with just those words.

In that sense, I will compute the accuracy of the 'about' tags in the data as follows:
$$
\text{Accuracy}_k = \frac{1}{n}\sum_{i=1}^n \mathbf{1} \{i \in R_k(q_i)\}
$$

Where:
- $q_i$: query built from the about tag(s) for document $i$. This query is made by concatenating all the 'about' tags.
- $R_k(q_i)$: the set of top-$k$ retrieved document IDs from ChromaDB for query $q_i$


This is implemented in the `ChromaClient` class with the function `document_accuracy_at_k`

In [28]:
from pathlib import Path
from src.embeds import TFIDFEmbedd, ChromaClient
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction 

In [29]:
# Creating the Chroma service 
CHROMA_PATH = Path("chroma_db/")
if not CHROMA_PATH.exists():
    CHROMA_PATH.mkdir(parents=True)

In [30]:
# English corpus
ids = [doc[0] for doc in corpus_en]
desc_en = [doc[1] for doc in corpus_en]
text_en = [doc[2] for doc in corpus_en]
metadata_en = [{'id': doc[0], 'about': list(doc[3])} for doc in corpus_en]

In [31]:
transformer_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
embedding_desc_fn = TFIDFEmbedd(desc_en)
embedding_text_fn = TFIDFEmbedd(text_en)

/home/canun/projects/IADB/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 776.05it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
chroma_client = ChromaClient(CHROMA_PATH)

chroma_client.create_collection(collection_name="descriptions_tfidf", embedding_fn=embedding_desc_fn)
chroma_client.add_documents(collection_name="descriptions_tfidf", id_list=ids, texts=desc_en, metadata=metadata_en)

chroma_client.create_collection(collection_name="texts_tfidf", embedding_fn=embedding_text_fn)
chroma_client.add_documents(collection_name="texts_tfidf", id_list=ids, texts=text_en, metadata=metadata_en)

In [33]:
chroma_client.create_collection(collection_name="descriptions_transf", embedding_fn=transformer_fn)
chroma_client.add_documents(collection_name="descriptions_transf", id_list=ids, texts=desc_en, metadata=metadata_en)

chroma_client.create_collection(collection_name="texts_transf", embedding_fn=transformer_fn)
chroma_client.add_documents(collection_name="texts_transf", id_list=ids, texts=text_en, metadata=metadata_en)

In [34]:
final_result = {}
for k in range(1,10,2):
    final_result[k] = {}
    for collection in ["descriptions_tfidf", "texts_tfidf", "descriptions_transf", "texts_transf"]:
        final_result[k][collection] = chroma_client.corpus_accuracy_at_k(collection_name = collection, docs = metadata_en, k=k)

final_result

{1: {'descriptions_tfidf': 0.5140845070422535,
  'texts_tfidf': 0.5422535211267606,
  'descriptions_transf': 0.7535211267605634,
  'texts_transf': 0.6267605633802817},
 3: {'descriptions_tfidf': 0.7112676056338029,
  'texts_tfidf': 0.7676056338028169,
  'descriptions_transf': 0.9366197183098591,
  'texts_transf': 0.8943661971830986},
 5: {'descriptions_tfidf': 0.823943661971831,
  'texts_tfidf': 0.8732394366197183,
  'descriptions_transf': 0.9647887323943662,
  'texts_transf': 0.9295774647887324},
 7: {'descriptions_tfidf': 0.852112676056338,
  'texts_tfidf': 0.9295774647887324,
  'descriptions_transf': 0.971830985915493,
  'texts_transf': 0.9577464788732394},
 9: {'descriptions_tfidf': 0.8802816901408451,
  'texts_tfidf': 0.9507042253521126,
  'descriptions_transf': 0.971830985915493,
  'texts_transf': 0.9577464788732394}}

Optimization

In [ ]:
from transformers import pipeline

# Load a multi-class classification pipeline
# if the model runs on CPU, comment out "device"
classifier = pipeline("text-classification", model="classla/ParlaCAP-Topic-Classifier", device=0, max_length=512, truncation=True)

# Example texts to classify
texts = [
    """I engage regularly with the CPS, and we recognise that this issue is a growing national priority.
      Prosecution rates have been rising year on year for knife crime.
      Between 2013-14 and 2017-18, there has been a 33% increase.
      The Offensive Weapons Bill now making its way through this House will tighten the law around the sale, delivery and possession of knives.""",
    """I appreciate that there are pressures in the hon. Gentleman’s constituency.
      I think most hon. Members would say that there are pressures in their constituency when it comes to general practice,
      so what have we done so far? Let me put it that way.
      This year, 3,157 medical school graduates will go on to specialise in general practice,
      which is the highest ever, but we still have to do more to improve the retention of GPs who are approaching retirement."""]

# Classify the texts
results = classifier(texts)

# Output the results
for result in results:
    print(result)

## Output
##{'label': 'Law and Crime', 'score': 0.9945019483566284}
##{'label': 'Health', 'score': 0.9890311360359192}
